# StorageGuard: finish Llama 3.1 8B

Re-runs only the 240 Llama episodes that crashed (DECISIONS.md, C15); everything finished is restored from Drive and skipped.

Before you start: upload the new `storageguard_colab.zip` to `MyDrive/storageguard/` (**Replace existing file**), pick **Runtime -> Change runtime type -> A100 GPU**, then **Runtime -> Run all** (about 30-40 min).

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "NO GPU")
assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU"
if "A100" not in gpu and "H100" not in gpu:
    print("WARNING: not an A100/H100. 7-8B models still fit on a 24 GB GPU (L4), but expect it to be slower.")

In [ ]:
BACKEND   = "vllm"
LLAMA     = "llama3.1-8b"
DRIVE_DIR = "/content/drive/MyDrive/storageguard"   # where storageguard_colab.zip is

import os, sys, time, zipfile, pathlib, subprocess
from google.colab import drive, userdata

drive.mount("/content/drive")
bundles = sorted(pathlib.Path(DRIVE_DIR).glob("storageguard_colab*.zip"), key=os.path.getmtime)
assert bundles, f"Upload storageguard_colab.zip to {DRIVE_DIR} first"
bundle = bundles[-1]                          # the newest upload, also if Drive kept an older copy
zipfile.ZipFile(bundle).extractall("/content")  # always the newest code; results are never in the zip
print("code from", bundle.name, "uploaded", time.strftime("%Y-%m-%d %H:%M", time.localtime(bundle.stat().st_mtime)))
os.chdir("/content/storageguard")
os.environ["SG_DRIVE_STATE"] = f"{DRIVE_DIR}/state"
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except Exception as exc:  # secret missing or notebook access not granted
    print(f"No HF_TOKEN ({type(exc).__name__}). Qwen and Mistral will work; Llama downloads will fail.")

def run(*args):
    """Run the driver, stream its output here, and stop the notebook if it fails."""
    p = subprocess.Popen([sys.executable, "-u", "colab/sg_colab.py", *map(str, args)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"step failed with exit code {p.returncode}; read the messages above")

def show_report():
    from IPython.display import Markdown, Image, display
    display(Markdown(pathlib.Path("results/REPORT.md").read_text()))
    for png in sorted(pathlib.Path("paper/figures").glob("*.png")):
        print(png.name)
        display(Image(filename=str(png), width=900))

def download_results():
    run("pack")   # zips results/, paper/, data/bench and configs into DRIVE_DIR
    from google.colab import files
    latest = max(pathlib.Path(DRIVE_DIR).glob("storageguard_results_*.zip"), key=os.path.getmtime)
    print("saved to Drive:", latest)
    files.download(str(latest))

print("project ready at", os.getcwd())

In [ ]:
run("setup", "--backend", BACKEND)

In [ ]:
run("all", "--backend", BACKEND, "--models", LLAMA, "--main-runs", 3, "--smoke-first")

In [ ]:
show_report()
download_results()